In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
)
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from datasets import load_dataset
from bio_agent.utils import *

In [2]:
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id="QuantFactory/Qwen2.5-0.5B-GGUF",
    filename="Qwen2.5-0.5B.Q4_K_M.gguf",
)

print(model_path)

/home/zaid/.cache/huggingface/hub/models--QuantFactory--Qwen2.5-0.5B-GGUF/snapshots/808e875628eb5b8d1e30728425039f4067b6ae16/Qwen2.5-0.5B.Q4_K_M.gguf


## fine-tuning

In [3]:
MODEL_ID = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(MODEL_ID)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [4]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)

In [5]:
model_1 = get_peft_model(model, lora_config)

In [6]:
dataset = load_dataset(
    "json",
    data_files="../datasets/summarizer-0.0.1/train.jsonl"
)
d1=dataset['train'][1]

In [7]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token



In [8]:
sample = tokenize(d1)

print(tokenizer.decode(sample["input_ids"]))
print(sample["labels"])

User: What is the function of the BRCA1 gene?
Assistant: The BRCA1 gene encodes a protein involved in DNA repair, specifically the repair of double-strand breaks. It plays a critical role in maintaining genomic stability. Mutations in BRCA1 are associated with an increased risk of breast and ovarian cancers.<|endoftext|>
[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 785, 18803, 5049, 16, 14862, 3209, 2539, 264, 12833, 6398, 304, 15552, 12733, 11, 11689, 279, 12733, 315, 1990, 41628, 437, 18303, 13, 1084, 11088, 264, 9023, 3476, 304, 20337, 79964, 19753, 13, 31228, 804, 304, 18803, 5049, 16, 525, 5815, 448, 458, 7172, 5214, 315, 17216, 323, 84442, 50323, 13, 151643]


In [9]:
tokenized_dataset = dataset.map(
    tokenize,
    remove_columns=dataset['train'].column_names,
)
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 55
    })
})

In [10]:
loader = DataLoader(
    dataset=tokenized_dataset['train'],
    batch_size=10,
    collate_fn=collate_fn
)

In [11]:
optimizer = torch.optim.Adam(params=model.parameters(),lr=3e-4)
loss = torch.nn.CrossEntropyLoss(ignore_index=-100)

In [12]:
NUM_EPOCHS = 4

model_1.train()

for epoch in range(NUM_EPOCHS):

    total_loss = 0.0

    for batch in loader:
        print("===")
        input_ids = batch["input_ids"].to(model_1.device)
        attention_mask = batch["attention_mask"].to(model_1.device)
        labels = batch["labels"].to(model_1.device)

        outputs = model_1(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        loss = outputs.loss

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}: "
        f"{total_loss / len(loader):.4f}"
    )

===
===
===
===
===
===
Epoch 1: 1.5474
===
===
===
===
===
===
Epoch 2: 1.2412
===
===
===
===
===
===
Epoch 3: 1.0464
===
===
===
===
===
===
Epoch 4: 0.8805


In [13]:
model_1.save_pretrained("../_checkpoints/qwen2.5-0.5b/summarizer-0.0.1")
tokenizer.save_pretrained("../_checkpoints/qwen2.5-0.5b/summarizer-0.0.1")

('../_checkpoints/qwen2.5-0.5b/summarizer-0.0.1/tokenizer_config.json',
 '../_checkpoints/qwen2.5-0.5b/summarizer-0.0.1/chat_template.jinja',
 '../_checkpoints/qwen2.5-0.5b/summarizer-0.0.1/tokenizer.json')

### ORIGINAL MODEL -- overriden by finetuned model

In [16]:
model.eval()
prompt = "User: What is a protein?\nAssistant: "
inputs = tokenizer(
    prompt,
    return_tensors="pt"
)


generated = model.generate(
    **inputs,
    max_new_tokens=128,
)
response = tokenizer.decode(
    generated[0],
    skip_special_tokens=True
)

print(response)

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


User: What is a protein?
Assistant: A protein is a large, complex molecule that performs various functions in living organisms. It is made up of amino acids arranged in a specific sequence to form an active protein. Proteins are responsible for many tasks, such as transporting substances, signaling, and maintaining cell structure.


### Fine tuned model

In [19]:
prompt = "User: Can you find recent research articles on CRISPR genome editing?\nAssistant: "
def ask_model(prompt=prompt, model=model_1):
    model.eval()
    
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )
    
    
    generated = model_1.generate(
        **inputs,
        max_new_tokens=128,
    )
    response = tokenizer.decode(
        generated[0],
        skip_special_tokens=True
    )
    
    print(response)
    return response

ask_model(model=model_1)

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


User: Can you find recent research articles on CRISPR genome editing?
Assistant: 1. "CRISPR-mediated gene editing: a review" (2023) - This review provides an overview of the CRISPR-Cas9 technology and its applications in various fields, including agriculture and medicine.
2. "CRISPR-based therapies for genetic diseases" (2023) - This article highlights the use of CRISPR in treating genetic disorders, focusing on clinical trials and potential future applications.
3. "CRISPR in precision medicine: current status and future directions" (2023) - This paper discusses the current state of research on CRISPR in precision medicine, including


'User: Can you find recent research articles on CRISPR genome editing?\nAssistant: 1. "CRISPR-mediated gene editing: a review" (2023) - This review provides an overview of the CRISPR-Cas9 technology and its applications in various fields, including agriculture and medicine.\n2. "CRISPR-based therapies for genetic diseases" (2023) - This article highlights the use of CRISPR in treating genetic disorders, focusing on clinical trials and potential future applications.\n3. "CRISPR in precision medicine: current status and future directions" (2023) - This paper discusses the current state of research on CRISPR in precision medicine, including'